# FIT5226 Project Stage 1: Tabular Q-Learning for a Single Agent
**Student Name:** Juan Pulido  
**Student ID:** 34169202  
**Student Email:** jpul0007@student.monash.edu  


# 📖 Table of Contents

## Phase I: Foundation
1. AI Declaration
2. Solution Architecture
3. Mathematical Formulation
## Phase II: Implementation
4. Configuration Layer
5. Environment Layer
6. Agent Layer
## Phase III: Training
7. Visualisation Engine
8. Training Loop
9. Model Training
10. Parameter Tuning
## Phase IV: Evaluation Layer
11. Evaluation Setup
12. Proof of Optimality
13. Path Comparison
14. Policy Vector Field
## Phase V: Conclusion
15. Interpretation and Limitations
---


# Phase I: Foundation

## 1. AI Declaration

In accordance with the unit's Generative AI policy, I declare that Generative AI (specifically Gemini) was used in the preparation of this notebook as an Architectural Co-Pilot, Technical Auditor, and coding assistant for non-core utility functions. 

Specifically, AI was used to assist with the following components:
*   **Visualisation Codebase:** Gemini assisted in writing the `matplotlib` syntax for the visualisation engine to generate the learning curves, agent trajectories, and vector fields.
*   **Architectural Design and Modularisation:** Used as a sounding board to transition my initial functional scripts into a cleaner, decoupled architecture (e.g., separating the reward logic from the environment physics).
*   **Validation Strategy and Evaluation:** Assisted in writing the unit and integration test cases to validate the underlying environment and agent logic, as well as structuring the exhaustive evaluation suite.
*   **Literate Programming and Mathematics:** Assisted in formatting LaTeX mathematical equations and refining semantic naming conventions for improved readability.

**Verification Statement:**
I have manually reviewed and evaluated all code provided by AI. I confirm that the core algorithmic logic (specifically the Python implementation of the **Bellman Optimality Equation**, the TD-target, and TD-error calculations), was manually written and mathematically verified against the unit lectures. I take full responsibility for the accuracy and integrity of this submission.


## 2. Solution Architecture
Before diving into the mathematical formulation and code, this section provides a high-level overview of the assignment solution. To ensure the codebase is robust, easily generalisable to different grid sizes, and adaptable to new reward structures, the solution was designed using a decoupled, modular architecture.

The implementation is partitioned into four primary layers:
1. **Configuration Layer:** Centralises all hyperparameters to eliminate "magic numbers" and make the agent and environment easily tunable.
2. **Environment Layer:** Manages the physics of the grid, boundaries, and payload mechanics. The reward logic is decoupled into its own class so penalty structures can be modified independently.
3. **Agent Layer:** Encapsulates the core Reinforcement Learning (RL) logic. The agent is entirely blind to the grid's underlying code, meaning it learns purely by receiving state observations and scalar rewards.
4. **Visualisation and Evaluation Layer:** A dedicated engine that safely extracts metrics during training to automatically generate performance comparisons and prove the optimal policy.

```text
+-------------------------------------------------------+
|                 Configuration Layer                   |
|         (Hyperparameters, Grid Dimensions)            |
+--------------------------+----------------------------+
             |             |                             
             v             v                             
+------------------+       +----------------------------+
|   Agent Layer    | <---  |     Environment Layer      |
| (Tabular Q-Agent)|  ---  | (GridWorld & Reward Logic) |
+---------+--------+   |   +-------------+--------------+
          |            |                 |               
          |  Action ---+---> State/Reward|               
          |                              |               
          +--------------+---------------+               
                         |                               
                         v                               
+-------------------------------------------------------+
|          Visualisation & Evaluation Layer             |
|        (Learning Curves, Policy Paths, Proofs)        |
+-------------------------------------------------------+
```

This architecture was chosen for the following reasons:

*   **Separation of Concerns:** By isolating the reward logic from the environment physics, we can test different incentive structures without risking the integrity of the state transition model.
*   **Model-Free Independence:** The Agent Layer does not know it is in a grid. It only knows it receives a state-tuple and must return an action-index.
*   **Automated Validation:** The dedicated Evaluation layer ensures that performance metrics are calculated independently of the training loop.

## 3. Mathematical Formulation
We formalise this transport task as a finite Markov Decision Process (MDP) defined by the tuple $(S, A, P, R, \gamma)$.

* **State Space ($S$):** The environment is fully observable. A state $s \in S$ consists of the agent's coordinates, the pickup location $A$, and a boolean payload flag. Mathematically, the upper bound of the state space size for a 5x5 grid is $|S| = 25 \times 25 \times 2 = 1250$, making a tabular approach computationally feasible.
* **Action Space ($A$):** A discrete set of 8 directional actions (NORTH, SOUTH, EAST, WEST, NORTH_EAST, NORTH_WEST, SOUTH_EAST, SOUTH_WEST).
* **Transition Dynamics ($P$):** The environment is deterministic. Given an action $a$ in state $s$, the probability of transitioning to state $s'$ is:
  $$P_{s,s'}^a = \begin{cases} 1 & \text{if } s' \text{ is the valid result of action } a \\ 0 & \text{otherwise} \end{cases}$$
* **Reward Function ($R$):** The expected immediate reward is defined as $R_s^a = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$.
* **Discount Factor ($\gamma$):** Set to $\gamma = 0.99$ to prioritise long-term returns while ensuring the mathematical convergence of infinite episodes.

**Goal:** The agent's objective is to maximise the **Expected Return** ($G_t$), defined as the discounted sum of future rewards: 
$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$ 
By applying a constant step cost of $R = -1.0$, the only way to maximise $G_t$ is to effectively minimise the number of steps taken to reach the terminal state.

# Phase II: Implementation

## 4. Configuration Layer

### 4.0 Library Installations

To ensure complete reproducibility and avoid any kernel conflicts, this solution was developed in an isolated Conda environment. If you wish to replicate this exact environment, please execute the following commands in your terminal before running the notebook:

```bash
conda create -n fit5226_ass1 python=3.14
conda activate fit5226_ass1
conda install ipykernel
```

Then, install required libraries for the assignment:

In [ ]:
%pip install -q numpy matplotlib

### 4.1 Global Imports and Seeding
We begin by importing the standard Python libraries alongside `numpy` for matrix operations and `matplotlib` for visualisation, adhering to the unit's permitted libraries. Global random seeds are set to guarantee that our evaluations and statistical proofs are fully reproducible.


In [ ]:
# Allowed libraries for the assignment
import numpy as np
import matplotlib.pyplot as plt

# Standard distribution of Python
from collections import defaultdict
from dataclasses import dataclass, field
import dataclasses
from enum import IntEnum
from typing import Tuple, List, Any, Optional, Dict
import random
import time

# Set seeds for reproducibility across all evaluations
np.random.seed(42)
random.seed(42)

### 4.2 Configuration Classes
To eliminate "magic numbers" and ensure the implementation is easily generalisable to different grid sizes or reward structures, we centralise all system hyperparameters into immutable Python `dataclasses`. This cleanly separates the experimental parameters from the core logic.

In [ ]:
@dataclass(frozen=True)
class AgentConfig:
    """
    Hyperparameters for RL Agents.
    Serves as the centralized registry for agent dynamics.
    
    Attributes:
        learning_rate_alpha (α): Step size for Q-table updates.
        discount_factor_gamma (γ): Importance of future rewards.
        initial_epsilon: Starting exploration probability.
        epsilon_decay_rate: Annealing rate for exploration.
        minimum_epsilon: Floor value for exploration.
        action_size: Size of the discrete action space (A).
    """
    learning_rate_alpha: float = 0.1
    discount_factor_gamma: float = 0.99
    initial_epsilon: float = 1.0
    epsilon_decay_rate: float = 0.9995  # Targeted for convergence < 15,000 episodes
    minimum_epsilon: float = 0.01
    action_size: int = 8

@dataclass(frozen=True)
class EnvConfig:
    """
    Parameters for the GridWorld environment.
    Serves as the centralized registry for environment physics and rewards.
    """
    grid_rows: int = 5
    grid_cols: int = 5
    step_cost: float = -1.0
    success_reward: float = 100.0

@dataclass(frozen=True)
class ExperimentConfig:
    """
    Global configuration for a training run.
    Aggregates all system constants to prevent magic numbers.
    """
    agent: AgentConfig = field(default_factory=AgentConfig)
    env: EnvConfig = field(default_factory=EnvConfig)
    training_episode_budget: int = 15000  # Strictly under the 20,000 limit

## 5. Environment Layer

### 5.1 The Action Space ($A$)
The agent possesses a discrete set of 8 actions mapping to cardinal and ordinal directions. To ensure semantic readability, we map these abstract action indices to physical 2D grid transitions (unit vectors).

The problem is formalised as a finite **Markov Decision Process (MDP)**.
*   **State Space ($S$):** Represented as a tuple: `((agent_row, agent_col), (pickup_row, pickup_col), has_payload)`.
*   **Action Space ($A$):** 8 discrete actions representing cardinal and ordinal movement.
*   **Transition Dynamics ($P$):** Deterministic. Boundary collisions result in the agent remaining in its current cell.

In [ ]:

class Directions(IntEnum):
    """
    Semantic enumeration of the Action Space (A).
    Maps cardinal and ordinal directions to discrete action indices.
    """
    WEST = 0; EAST = 1; SOUTH = 2; NORTH = 3
    NORTH_WEST = 4; NORTH_EAST = 5; SOUTH_WEST = 6; SOUTH_EAST = 7

    @property
    def coordinate_delta(self) -> Tuple[int, int]:
        return Actions.map_direction_to_unit_vector(self)

class Actions:
    """
    Utility class for manipulating the Action Space and coordinate geometry.
    Maps the abstract action indices to physical grid transitions.
    """
    _DIRECTION_TO_VECTOR_MAP = {
        Directions.WEST: (-1, 0), Directions.EAST: (1, 0),
        Directions.SOUTH: (0, -1), Directions.NORTH: (0, 1),
        Directions.NORTH_WEST: (-1, -1), Directions.NORTH_EAST: (-1, 1),
        Directions.SOUTH_WEST: (1, -1), Directions.SOUTH_EAST: (1, 1),
    }

    @staticmethod
    def map_direction_to_unit_vector(direction: Directions) -> Tuple[int, int]:
        """
        Converts a Directions enum to a (dx, dy) unit vector tuple.
        """
        return Actions._DIRECTION_TO_VECTOR_MAP[direction]

    @staticmethod
    def calculate_next_coordinates(current_coordinates: Tuple[int, int], action_index: int) -> Tuple[int, int]:
        """
        Computes the next state coordinates after applying an action.
        """
        dx, dy = Actions._DIRECTION_TO_VECTOR_MAP[Directions(action_index)]
        return (current_coordinates[0] + dx, current_coordinates[1] + dy)

    @staticmethod
    def is_within_boundaries(coordinates: Tuple[int, int], rows: int, cols: int) -> bool:
        """
        Validates if a coordinate pair exists within the defined grid boundaries.
        """
        x, y = coordinates
        return 0 <= x < rows and 0 <= y < cols

### 5.2 The Reward Structure ($R$)
The reward signal is decoupled into its own class to allow dynamic penalty adjustments without altering the grid physics. 

We mathematically incentivise the shortest path by applying a constant **Step Penalty ($R = -1.0$)**. To maximise the Return $G_t$, the agent is forced to minimise the total steps taken. Reaching the terminal state B with the payload yields a **Success Reward ($R = +100.0$)**, which acts as a strong positive anchor for the Bellman updates.

The reward structure is designed to incentivise the **shortest possible path**:
1.  **Step Penalty ($R = -1.0$):** Every action taken yields a negative reward. This ensures that the agent is penalised for every additional move, forcing it to find the shortest path.
2.  **Success Reward ($R = +100.0$):** Awarded upon reaching location B while carrying the item.

This design creates a sparse but clear gradient: the agent's total return $G_t$ is maximised only by minimising the total steps taken.

In [ ]:
class TransportReward:
    """
    Concrete reward implementation for transport tasks.
    Incentivizes efficiency via step costs and rewards delivery success.
    """
    def __init__(self, config: EnvConfig = EnvConfig()):
        self.config = config

    def calculate_reward(self, previous_state: Tuple[Any, ...], action: int, 
                         current_state: Tuple[Any, ...], is_terminal_state: bool) -> float:
        """
        Calculates the scalar reward signal R_{t+1}.
        
        Returns:
            - success_reward if the agent reached the goal.
            - step_cost otherwise (to penalise long paths).
        """
        if is_terminal_state:
            return self.config.success_reward
        return self.config.step_cost

### 5.3 The Environment Dynamics ($P$) and State Space ($S$)
The `TransportGridWorld` manages the deterministic transition dynamics ($P_{s,s'}^a = 1$). If an action drives the agent into a grid boundary, the agent remains in its current cell. 

A state $s \in S$ is fully observable and represented as the tuple: `(agent_coordinates, pickup_coordinates, has_payload)`. The environment automatically handles the payload pickup at $A$ and the terminal discharge at $B(n,n)$.

In [ ]:
class TransportGridWorld:
    """
    Rectangular grid-based environment for pickup and delivery transport tasks.
    
    This class models the environment dynamics, state transitions, and 
    physical constraints of the world. It provides states to the agent
    and executes actions to transition between states.
    """
    def __init__(self, config: EnvConfig = EnvConfig(), reward_provider: Optional[TransportReward] = None):
        """
        Initializes the world with generic MxN dimensions.
        
        Args:
            config: EnvConfig object containing dimensions and reward values.
            reward_provider: Concrete implementation of BaseReward.
        """
        self.config = config
        
        # Fixed Goal B is at the bottom-right corner: (M-1, N-1)
        self.goal_coordinates = (self.config.grid_rows - 1, self.config.grid_cols - 1)
        
        # Dependency Injection for Reward Logic
        self.reward_provider = reward_provider or TransportReward(config=self.config)
        self.reset()

    def reset(self) -> Tuple[Any, ...]:
        """
        INITIALISATION:
        Resets the world to a new starting configuration for a fresh episode.
        """
        self.agent_coordinates = (np.random.randint(self.config.grid_rows), np.random.randint(self.config.grid_cols))
        while True:
            self.pickup_coordinates = (np.random.randint(self.config.grid_rows), np.random.randint(self.config.grid_cols))
            if self.pickup_coordinates != self.goal_coordinates:
                break
        self.has_payload = False
        return self.get_state()

    def get_state(self) -> Tuple[Any, ...]:
        """
        PERCEPTION
        Constructs the state representation provided to the agent (S_t).
        
        Returns:
            A tuple of (agent_pos, pickup_pos, has_payload)
        """
        return (self.agent_coordinates, self.pickup_coordinates, self.has_payload)

    def execute_step(self, action_idx: int) -> Tuple[Tuple[Any, ...], float, bool]:
        """
        DYNAMICS (State Transition)
        Executes an action and calculates the resulting environment state.
        
        Args:
            action_idx: Index representing the direction of movement.
            
        Returns:
            next_state: The new state (S_{t+1})
            reward: The scalar feedback signal (R_{t+1})
            is_terminal_state: Whether the goal was reached.
        """
        # Store the previous state for reward calculation
        state_before_action = self.get_state()

        # Transition Logic: Compute next potential position
        potential_next_position = Actions.calculate_next_coordinates(self.agent_coordinates, action_idx)

         # Validation: Check if the move is legal (within grid boundaries)
        if Actions.is_within_boundaries(potential_next_position, self.config.grid_rows, self.config.grid_cols):
            self.agent_coordinates = potential_next_position

        # Mechanics: Automatic pickup upon entering the pickup coordinates
        if not self.has_payload and self.agent_coordinates == self.pickup_coordinates:
            self.has_payload = True

        # Mechanics: Check for termination (delivery at goal B)
        is_terminal_state = self.has_payload and self.agent_coordinates == self.goal_coordinates
        
        # Capture resulting state
        next_state = self.get_state()
        
        # REWARD: Compute feedback signal using the injected reward provider
        reward = self.reward_provider.calculate_reward(state_before_action, action_idx, next_state, is_terminal_state)

        return next_state, reward, is_terminal_state

## 6. Agent Layer

### Model-Free Control and Temporal Difference (TD) Learning
Because the agent does not possess a prior model of the transition dynamics ($P$), it cannot solve the MDP using standard Dynamic Programming (Value Iteration). Instead, it must use **Model-Free Control** to approximate the optimal action-value function $q_*(s,a)$ through trial and error.

The theoretical foundation is the **Bellman Optimality Equation** for Action-Values:
$$q_{*}(s,a) = R_{s}^{a} + \gamma \sum_{s' \in S} P_{s,s'}^{a} \max_{a' \in A} q_{*}(s',a')$$

Since $P_{s,s'}^{a}$ is unknown, **Q-Learning** approximates this using **Temporal-Difference (TD) sampling**. It replaces the true expected future value with a sample backup obtained after taking a single step. The off-policy TD control update rule is:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha [ \underbrace{R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a')}_{\text{TD Target}} - Q(S_t, A_t) ]$$

* **TD Target:** The updated estimate based on the immediate reward and the greedy evaluation of the next state.
* **TD Error:** The difference between the TD Target and the current estimate $Q(S_t, A_t)$, scaled by the learning rate $\alpha$.

### GLIE Convergence Guarantee
To mathematically guarantee that the Q-table converges to the optimal $q_*(s,a)$, the agent must satisfy the **Greedy in the Limit with Infinite Exploration (GLIE)** conditions:
1. All state-action pairs must be visited infinitely often.
2. The policy must eventually converge to a greedy policy.

We implement an $\epsilon$-greedy policy where the exploration probability $\epsilon$ decays after each terminal state: 
$$\epsilon_{t+1} = \epsilon_t \times \text{decay\_rate}$$ 
As $\epsilon \rightarrow 0$, the policy smoothly transitions from stochastic exploration to pure exploitation of the learned values.

In [ ]:
class TabularQAgent:
    """
    Agent implementation using the Tabular Q-Learning algorithm.
    
    This agent learns an optimal policy by estimating the state-action value 
    function Q(s, a) through interaction with the environment. It uses 
    Temporal Difference (TD) learning to update its knowledge.
    """
    def __init__(self, config: AgentConfig):
        """
        Initializes the agent with a centralized configuration.
        
        Args:
            config: AgentConfig object containing hyperparameters.
        """
        self.config = config
        
        # Initialise to a vector of zeros representing all possible actions
        self.action_value_table = defaultdict(lambda: np.zeros(self.config.action_size))
        
        # Exploration rate (epsilon) for the epsilon-greedy strategy
        self.current_exploration_probability = self.config.initial_epsilon

    def select_action(self, current_state: Tuple[Any, ...], use_greedy: bool = False) -> int:
        """
        ACTION SELECTION (Epsilon-Greedy Policy)
        
        Mathematically:
        π(s) = { random action from A         with probability ε
               { argmax_a Q(s, a)             with probability 1 - ε
        
        Args:
            current_state: The state the agent is currently in (s_t).
            use_greedy: If True, the agent exploits only (for evaluation).
        """
        # Exploration: Uniformly random action selection
        if not use_greedy and np.random.rand() < self.current_exploration_probability:
            return int(np.random.randint(self.config.action_size))
        
        # Exploitation: Optimal action selection based on current knowledge
        # vectorised argmax over the action-value array for the current state.
        return int(np.argmax(self.action_value_table[current_state]))

    def apply_learning_step(self, 
                            previous_state: Tuple[Any, ...], 
                            action: int, 
                            reward: float, 
                            current_state: Tuple[Any, ...], 
                            is_terminal_state: bool) -> None:
        """
        LEARNING (Bellman Optimality Update)
        
        This method implements the core Temporal Difference (TD) learning 
        using the Tabular Q-Learning update rule:
        
        Q(s, a) ← Q(s, a) + α * [R + γ * max_a' Q(s', a') - Q(s, a)]
        
        Where:
            α (alpha) is the learning rate.
            γ (gamma) is the discount factor.
            R is the reward received after taking action a in state s.
            max_a' Q(s', a') is the maximum Q-value for the next state s'.
        
        Args:
            previous_state (s): The state before the action was taken (s_t).
            action (a): The action executed (a_t).
            reward (R): The feedback signal received (R_{t+1}).
            current_state (s'): The resulting state after the action (s_{t+1}).
            is_terminal_state: Boolean indicating if the transition led to a goal.
        """
        # Retrieve Current Estimate Q(s, a)
        # Access the Q-table for the previous state and specific action taken.
        current_q_value = self.action_value_table[previous_state][action]
        
        # Compute estimated optimal future value max_a' Q(s', a')
        # If the next state is terminal, the future reward is zero by definition.
        max_future_q = 0.0 if is_terminal_state else np.max(self.action_value_table[current_state])
        
        # Computhe the TD Target: R + γ * max_a' Q(s', a')
        td_target = reward + (self.config.discount_factor_gamma * max_future_q)

         # Compute the TD Error (Temporal Difference): Target - current estimate
        td_error = td_target - current_q_value
        
        # Update Q-Table: Q(s,a) ← Q(s,a) + α * TD_Error
        # This is the foundational Bellman update for off-policy Q-Learning.
        self.action_value_table[previous_state][action] += self.config.learning_rate_alpha * td_error

        # Anneal Exploration Rate (Epsilon Decay)
        # Decrease the probability of random actions as the agent gains more knowledge.
        if is_terminal_state:
            self.current_exploration_probability = max(
                self.config.minimum_epsilon, 
                self.current_exploration_probability * self.config.epsilon_decay_rate
            )

# Phase III: Training

## 7. Visualisation Engine
To satisfy the requirement for fully automated visualisations, we define a static `GridVisualiser` class. This engine wraps `matplotlib` to render learning curves, policy paths, and vector fields, ensuring that testing and analysis can be performed with minimal user interaction.

In [ ]:
class GridVisualiser:
    """Provides high-quality visualisations for the GridWorld environment."""
    
    @staticmethod
    def plot_policy_path(env: TransportGridWorld, path: List[Tuple[int, int]], title: str = "Agent Path", ax: Optional[plt.Axes] = None):
        """
        Renders the grid with the agent's path, source, and destination.
        
        Args:
            env: The TransportGridWorld environment instance.
            path: List of (row, col) tuples representing the agent's journey.
            title: Title for the plot.
        """
        rows, cols = env.config.grid_rows, env.config.grid_cols
        if ax is None: fig, ax = plt.subplots(figsize=(max(cols, 6), max(rows, 6)))
        
        ax.set_xticks(np.arange(-.5, cols, 1), minor=True)
        ax.set_yticks(np.arange(-.5, rows, 1), minor=True)
        ax.grid(which='minor', color='black', linestyle='-', linewidth=2)
        
        ax.plot(env.pickup_coordinates[1], env.pickup_coordinates[0], 'go', markersize=20, label='Pickup A', alpha=0.6)
        ax.plot(env.goal_coordinates[1], env.goal_coordinates[0], 'rs', markersize=20, label='Dropoff B', alpha=0.6)
        
        if path:
            y_coords, x_coords = zip(*path)
            ax.plot(x_coords, y_coords, 'b--', linewidth=3, label='Agent Path', alpha=0.8)
            ax.plot(x_coords[0], y_coords[0], 'b*', markersize=15, label='Start')
            ax.plot(x_coords[-1], y_coords[-1], 'ro', markersize=10)
        
        ax.set_title(title, fontsize=16); ax.set_xlim(-0.5, cols - 0.5); ax.set_ylim(rows - 0.5, -0.5)
        for r in range(rows):
            for c in range(cols): ax.text(c, r, f"({r},{c})", va='center', ha='center', color='gray', fontsize=8)

        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        if ax is None: plt.tight_layout(); return fig

    @staticmethod
    def plot_learning_curve(rewards: List[float], window: int = 100, title: str = "Learning Curve"):
        """
        Plots the moving average of rewards during training.
        
        Args:
            rewards: List of total rewards per episode.
            window: Moving average window size.
            title: Title for the plot.
        """
        plt.figure(figsize=(12, 6))
        if len(rewards) >= window:
            averages = np.convolve(rewards, np.ones(window)/window, mode='valid')
            plt.plot(averages, label=f"Moving Avg (w={window})", color='blue', linewidth=2)
        plt.plot(rewards, alpha=0.2, color='gray', label="Raw Rewards")
        plt.title(title, fontsize=16); plt.xlabel("Episode"); plt.ylabel("Total Reward"); plt.grid(True); plt.legend(); plt.show()

    @staticmethod
    def plot_performance_comparison(results: Dict[str, List[float]], window: int = 500, title: str = "Performance Comparison"):
        """
        Compares the learning progress of multiple agents or configurations.
        
        Args:
            results: Dictionary mapping label names to lists of rewards.
            window: Moving average window size.
            title: Title for the comparison plot.
        """
        plt.figure(figsize=(14, 7))
        for label, rewards in results.items():
            if len(rewards) >= window:
                averages = np.convolve(rewards, np.ones(window)/window, mode='valid')
                plt.plot(averages, label=label, linewidth=2)
        plt.title(title, fontsize=18); plt.xlabel("Episode"); plt.ylabel("Average Reward"); plt.grid(True); plt.legend(); plt.show()

    @staticmethod
    def visualise_policy_grid(env: TransportGridWorld, agent: TabularQAgent):
        """
        Renders a quiver plot showing the preferred action in each state.
        Only visualises the grid where has_payload=False for simplicity.
        """
        rows, cols = env.config.grid_rows, env.config.grid_cols
        U, V = np.zeros((rows, cols)), np.zeros((rows, cols))
        for r in range(rows):
            for c in range(cols):
                state = ((r, c), (2, 2), False) 
                action = agent.select_action(state, use_greedy=True)
                dx, dy = Actions.map_direction_to_unit_vector(Directions(action))
                U[r, c], V[r, c] = dx, dy
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.quiver(np.arange(cols), np.arange(rows), U, V, color='teal', pivot='mid', scale=20)
        ax.plot(2, 2, 'go', markersize=15, label='Pickup A (2,2)')
        ax.set_title("Learned Policy (Greedy Actions Toward A)", fontsize=16); ax.invert_yaxis(); plt.legend(); plt.show()

## 8. Training Loop
The `train_session` function acts as the primary orchestrator between the Agent and the Environment. It resets the grid, handles the $\epsilon$-greedy action selection, passes the scalar rewards to the agent's Bellman update function, and logs the cumulative rewards for convergence analysis.

In [ ]:
def train_session(agent, env, episodes):
    rewards_log = []
    print(f"[*] Training Agent for {episodes} episodes...")
    for _ in range(episodes):
        s = env.reset(); total_r = 0; done = False
        while not done:
            a = agent.select_action(s)
            s_p, r, done = env.execute_step(a)
            agent.apply_learning_step(s, a, r, s_p, done)
            s, total_r = s_p, total_r + r
        rewards_log.append(total_r)
    return rewards_log

## 9. Model Training
We instantiate our optimal agent using standard hyperparameters and train it for up to 15,000 episodes, plotting the learning curve to prove convergence.

In [ ]:
# Get hyperparameters from configuration layer
exp_config = ExperimentConfig()
env = TransportGridWorld(exp_config.env)
trained_agent = TabularQAgent(exp_config.agent)

# Run the 15,000 episode training block
rewards = train_session(trained_agent, env, exp_config.training_episode_budget)

# Visualise the convergence curve generated by training
GridVisualiser.plot_learning_curve(rewards, title="Optimal Agent Training Convergence")

## 10. Parameter Tuning
To demonstrate the impact of hyperparameters, we train two entirely separate, temporary agents with different learning rates ($\alpha$) to compare their convergence efficiency.

In [ ]:
comparison_results = {}
for alpha in [0.05, 0.5]:
    cfg = dataclasses.replace(exp_config.agent, learning_rate_alpha=alpha)
    cmp_agent = TabularQAgent(cfg)
    comparison_results[f"Alpha={alpha}"] = train_session(cmp_agent, TransportGridWorld(exp_config.env), 5000)
    
GridVisualiser.plot_performance_comparison(comparison_results, title="Impact of Learning Rate (Alpha)")

# Phase IV: Evaluation Layer

## 11. Evaluation Setup
According to the assignment specification, the agent must solve 100% of all possible scenarios in 10 steps or less, independently of the location of A. The function below aligns exactly with standard combinatorial iteration.

In [ ]:
TARGET_SUCCESS_RATE = 100.0      # Required percentage for outstanding grade
MAX_ALLOWED_STEPS = 10           # Efficiency limit for any scenario

def run_evaluation(agent: TabularQAgent, env: TransportGridWorld):
    """
    PERFORMANCE EVALUATION
    
    This function automates the exhaustive verification of the trained agent.
    It proves the policy learned is both robust (100% success) and 
    optimal (minimum steps), regardless of the pickup location A.
    """
    print("\n[*] Commencing Exhaustive Evaluation (All possible scenarios)...")
    rows = env.config.grid_rows
    cols = env.config.grid_cols
    goal = env.goal_coordinates
    
    success_count = 0
    total_scenarios = 0
    max_steps_observed = 0

    # Combinatoric Iteration: Every valid (Start, Pickup) coordinate pair
    for start_row in range(rows):
        for start_col in range(cols):
            for pickup_row in range(rows):
                for pickup_col in range(cols):
                    if (start_row, start_col) == goal or (pickup_row, pickup_col) == goal:
                        continue
                    
                    # Manually inject state for verification
                    env.agent_coordinates = (start_row, start_col)
                    env.pickup_coordinates = (pickup_row, pickup_col)
                    env.has_payload = False
                    state = env.get_state()
                    
                    done = False
                    steps = 0
                    
                    # Pure Exploitation: test the learned policy (use_greedy=True)
                    while not done and steps < 20:
                        action = agent.select_action(state, use_greedy=True)
                        state, _, done = env.execute_step(action)
                        steps += 1
                    
                    total_scenarios += 1
                    if done and steps <= MAX_ALLOWED_STEPS:
                        success_count += 1
                    max_steps_observed = max(max_steps_observed, steps)

    success_rate = (success_count / total_scenarios) * 100
    
    print("-" * 50)
    print(f"{'METRIC':<30} | {'VALUE':<15}")
    print("-" * 50)
    print(f"{'Total Unique Scenarios':<30} | {total_scenarios:<15}")
    print(f"{'Success Rate (<= 10 steps)':<30} | {success_rate:>14.2f}%")
    print(f"{'Maximum Steps Taken':<30} | {max_steps_observed:<15}")
    print("-" * 50)

    if success_rate == TARGET_SUCCESS_RATE and max_steps_observed <= MAX_ALLOWED_STEPS:
        print("\n\u2705 PASSED: All assignment requirements met!")
    else:
        print("\n\u274c FAILED: Performance constraints violated.")

## 12. Proof of Optimality
We test the pre-trained agent against every valid (Start, Pickup) coordinate pair on the $5 \times 5$ grid. It strictly measures the greedy policy ($\epsilon = 0$) to ensure the agent has truly internalised the optimal navigation path.

In [ ]:
# Evaluate the agent trained in Phase III
run_evaluation(trained_agent, env)

## 13. Path Comparison
**Untrained vs Trained:** To visually prove learning has occurred, we extract and compare the path of a completely random, untrained agent against our fully converged agent on an injected worst-case scenario grid.

In [ ]:
def extract_path(agent, env, start_coords, pickup_coords):
    env.agent_coordinates = start_coords
    env.pickup_coordinates = pickup_coords
    env.has_payload = False
    state = env.get_state()
    
    path = [env.agent_coordinates]
    done = False
    steps = 0
    while not done and steps < 20:
        action = agent.select_action(state, use_greedy=True)
        state, _, done = env.execute_step(action)
        path.append(env.agent_coordinates)
        steps += 1
    return path

# Set up a Worst-Case Scenario
worst_case_start = (0, 0)
worst_case_pickup = (4, 0)

# Extract path for Untrained Agent
untrained_agent = TabularQAgent(exp_config.agent)
untrained_path = extract_path(untrained_agent, env, worst_case_start, worst_case_pickup)

# Extract path for Trained Agent
trained_path = extract_path(trained_agent, env, worst_case_start, worst_case_pickup)

# Plot Comparison Side-by-Side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
GridVisualiser.plot_policy_path(env, untrained_path, title="Untrained Agent Path (Random)", ax=ax1)
GridVisualiser.plot_policy_path(env, trained_path, title="Trained Agent Path (Optimal)", ax=ax2)
plt.show()

## 14. Policy Vector Field
This Quiver plot maps the agent's internal Q-table as directional arrows across the entire grid, showing exactly how the agent intends to move from any state toward the pickup point.

In [ ]:
# Visualise General Policy Vector Field
GridVisualiser.visualise_policy_grid(env, trained_agent)

# Phase V: Conclusion

## 15. Interpretation and Limitations

### Policy Interpretation
The agent learned a navigation strategy that minimises **the total number of steps taken**. The vector field confirms that the agent correctly prioritises moving toward the pickup location before heading to the goal.

Because Q-Learning is a model-free TD control algorithm, the optimal action-value function $Q(s,a)$ successfully caches the value of the infinite future into a single matrix. Extracting the optimal policy $\pi_*$ requires no complex forward-search; the agent simply acts greedily with respect to the converged Q-table:
$$\pi_{*}(a|s) = \begin{cases} 1 & \text{if } a = \arg\max_{a' \in A} Q(s,a') \\ 0 & \text{otherwise} \end{cases}$$

This deterministic optimal policy successfully maps the shortest path regardless of the random starting coordinates, proving that the agent has learned the underlying manifold of the environment's state-space.

### Limitations
While Tabular methods perform flawlessly for this $5 \times 5$ environment, they do not scale to high-dimensional state spaces. As grid dimensions scale, the number of required Q-table entries grows exponentially. In continuous observation spaces or massive grids (e.g., $100 \times 100$), the state-action visitation frequency drops precipitously, making tabular convergence computationally intractable. In such industrial applications, table-based models must be abandoned in favour of non-linear Value Function Approximations, such as Deep Q-Networks (DQN), to overcome the memory bottleneck of a sparse Q-table.